In [1]:
!pip install pandas pyarrow


In [2]:
!pip install scikit-learn


In [18]:
import pandas as pd
import numpy as np
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
import pandas as pd
from pathlib import Path

In [10]:
PARQUET_PATH = r"D:\Master\IBD\traffic_full.parquet"

df = pd.read_parquet(PARQUET_PATH)

if "__null_dask_index__" in df.columns:
    df = df.drop(columns=["__null_dask_index__"])


In [11]:
numeric_cols = [
    "latitude", "longitude", "speed_limit",
    "weather_conditions", "road_surface_conditions", "light_conditions",
    "collision_severity",
    "vehicle_reference", "vehicle_type",
    "age_of_vehicle", "age_of_driver", "sex_of_driver",
    "engine_capacity_cc", "propulsion_code", "journey_purpose_of_driver",
    "casualty_reference", "casualty_class", "sex_of_casualty",
    "age_of_casualty", "casualty_severity",
]

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df = df.dropna(subset=["casualty_severity"]).copy()
df["casualty_severity"] = df["casualty_severity"].astype(int)

df["casualty_severity"].value_counts()


casualty_severity
3    9348205
2    1235185
1     109203
Name: count, dtype: int64

In [12]:
numeric_features = [
    "latitude", "longitude",
    "speed_limit",
    "age_of_vehicle", "age_of_driver", "engine_capacity_cc",
    "age_of_casualty",
]

categorical_features = [
    "weather_conditions",
    "road_surface_conditions",
    "light_conditions",
    "vehicle_type",
    "sex_of_driver",
    "propulsion_code",
    "journey_purpose_of_driver",
    "casualty_class",
    "sex_of_casualty",
]

feature_cols = numeric_features + categorical_features

def make_preprocess():
    num_tf = Pipeline([("imputer", SimpleImputer(strategy="median"))])
    cat_tf = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ])
    return ColumnTransformer(
        [("num", num_tf, numeric_features),
         ("cat", cat_tf, categorical_features)]
    )


In [13]:
fatal   = df[df["casualty_severity"] == 1]
serious = df[df["casualty_severity"] == 2]
slight  = df[df["casualty_severity"] == 3]

len(fatal), len(serious), len(slight)


(109203, 1235185, 9348205)

In [14]:
N_FATAL   = min(20_000, len(fatal))     
N_SERIOUS = min(40_000, len(serious))   
N_SLIGHT  = min(80_000, len(slight))    


In [15]:
fatal_train   = fatal.sample(N_FATAL, random_state=42)
serious_train = serious.sample(N_SERIOUS, random_state=42)
slight_train  = slight.sample(N_SLIGHT, random_state=42)

balanced_df = pd.concat(
    [fatal_train, serious_train, slight_train],
    ignore_index=True
)

balanced_df["casualty_severity"].value_counts()


casualty_severity
3    80000
2    40000
1    20000
Name: count, dtype: int64

In [19]:
train_df, val_df = train_test_split(
    balanced_df,
    test_size=0.2,
    random_state=42,
    stratify=balanced_df["casualty_severity"]
)

train_df["casualty_severity"].value_counts(), val_df["casualty_severity"].value_counts()


(casualty_severity
 3    64000
 2    32000
 1    16000
 Name: count, dtype: int64,
 casualty_severity
 3    16000
 2     8000
 1     4000
 Name: count, dtype: int64)

In [20]:
X_train = train_df[feature_cols]
y_train = train_df["casualty_severity"]

X_val = val_df[feature_cols]
y_val = val_df["casualty_severity"]


In [21]:
test_df = df.sample(100_000, random_state=123)   # 100k rows for test

X_test = test_df[feature_cols]
y_test = test_df["casualty_severity"]

y_test.value_counts()


casualty_severity
3    87467
2    11564
1      969
Name: count, dtype: int64

In [22]:
preprocess = make_preprocess()

# push the model to care more about Fatal and Serious
class_weights = {
    1: 6.0,   # Fatal
    2: 3.0,   # Serious
    3: 1.0,   # Slight
}

rf = RandomForestClassifier(
    n_estimators=400,      # more trees → better, slower
    max_depth=22,          # quite deep
    min_samples_leaf=10,   # smoother, avoids crazy splits
    random_state=42,
    n_jobs=-1,
    class_weight=class_weights,
)

model_rf3 = Pipeline([
    ("preprocess", preprocess),
    ("rf", rf),
])

model_rf3.fit(X_train, y_train)


,steps,"[('preprocess', ...), ('rf', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [23]:
y_val_pred = model_rf3.predict(X_val)
print("Validation set:")
print(classification_report(y_val, y_val_pred))
print(confusion_matrix(y_val, y_val_pred))


Validation set:
              precision    recall  f1-score   support

           1       0.31      0.66      0.42      4000
           2       0.35      0.52      0.42      8000
           3       0.80      0.38      0.52     16000

    accuracy                           0.46     28000
   macro avg       0.49      0.52      0.45     28000
weighted avg       0.60      0.46      0.48     28000

[[2658 1080  262]
 [2596 4129 1275]
 [3335 6517 6148]]


In [24]:
y_test_pred = model_rf3.predict(X_test)
print("Test set (real distribution):")
print(classification_report(y_test, y_test_pred))
print(confusion_matrix(y_test, y_test_pred))


Test set (real distribution):
              precision    recall  f1-score   support

           1       0.03      0.67      0.06       969
           2       0.14      0.51      0.22     11564
           3       0.95      0.39      0.55     87467

    accuracy                           0.40    100000
   macro avg       0.37      0.52      0.28    100000
weighted avg       0.84      0.40      0.51    100000

[[  650   249    70]
 [ 3827  5876  1861]
 [17790 35793 33884]]


In [25]:
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix

# 1) Probabilities for each class on the SAME X_test as before
proba_test = model_rf3.predict_proba(X_test)
classes = list(model_rf3.named_steps["rf"].classes_)  # should be [1, 2, 3]

idx_fatal   = classes.index(1)
idx_serious = classes.index(2)
idx_slight  = classes.index(3)

fatal_p   = proba_test[:, idx_fatal]
serious_p = proba_test[:, idx_serious]
slight_p  = proba_test[:, idx_slight]

# 2) Helper: prediction with custom thresholds
def predict_with_thresholds(T_FATAL=0.7, T_SERIOUS=0.5):
    """
    Use RF probabilities, but only say:
      1 (Fatal)   if P(Fatal)   >= T_FATAL
      2 (Serious) if P(Serious) >= T_SERIOUS
      otherwise   3 (Slight)
    """
    y_custom = np.full(len(X_test), 3, dtype=int)  # default Slight

    for i in range(len(X_test)):
        if fatal_p[i] >= T_FATAL:
            y_custom[i] = 1
        elif serious_p[i] >= T_SERIOUS:
            y_custom[i] = 2
        else:
            y_custom[i] = 3

    return y_custom

# quick example just to see something:
y_test_custom = predict_with_thresholds(T_FATAL=0.7, T_SERIOUS=0.5)
print("Custom thresholds example (T_FATAL=0.7, T_SERIOUS=0.5):")
print(classification_report(y_test, y_test_custom))
print(confusion_matrix(y_test, y_test_custom))


Custom thresholds example (T_FATAL=0.7, T_SERIOUS=0.5):
              precision    recall  f1-score   support

           1       0.24      0.05      0.08       969
           2       0.18      0.18      0.18     11564
           3       0.88      0.89      0.89     87467

    accuracy                           0.80    100000
   macro avg       0.44      0.38      0.39    100000
weighted avg       0.80      0.80      0.80    100000

[[   50    52   867]
 [   43  2136  9385]
 [  116  9391 77960]]


In [26]:
results = []

for T_F in [0.5, 0.6, 0.7, 0.8]:
    for T_S in [0.4, 0.5, 0.6]:
        y_c = predict_with_thresholds(T_FATAL=T_F, T_SERIOUS=T_S)
        rep = classification_report(y_test, y_c, output_dict=True, zero_division=0)
        results.append({
            "T_FATAL": T_F,
            "T_SERIOUS": T_S,
            "macro_f1": rep["macro avg"]["f1-score"],
            "fatal_f1": rep["1"]["f1-score"],
            "serious_f1": rep["2"]["f1-score"],
            "slight_f1": rep["3"]["f1-score"],
            "fatal_prec": rep["1"]["precision"],
            "fatal_rec": rep["1"]["recall"],
        })

res_df = pd.DataFrame(results).sort_values("macro_f1", ascending=False)
res_df.head(10)


,T_FATAL,T_SERIOUS,macro_f1,fatal_f1,serious_f1,slight_f1,fatal_prec,fatal_rec
4,0.6,0.5,0.393018,0.112270,0.184591,0.882193,0.079878,0.188854
7,0.7,0.5,0.385670,0.084890,0.184591,0.887528,0.239234,0.051600
1,0.5,0.5,0.377855,0.088820,0.184591,0.860153,0.049734,0.414861
5,0.6,0.6,0.364671,0.112270,0.059487,0.922256,0.079878,0.188854
10,0.8,0.5,0.364222,0.020243,0.184591,0.887831,0.526316,0.010320
3,0.6,0.4,0.362961,0.112270,0.227099,0.749514,0.079878,0.188854
8,0.7,0.6,0.357079,0.084890,0.059487,0.926861,0.239234,0.051600
6,0.7,0.4,0.356498,0.084890,0.227099,0.757504,0.239234,0.051600
2,0.5,0.6,0.350348,0.088820,0.059487,0.902735,0.049734,0.414861
0,0.5,0.4,0.344673,0.088820,0.226578,0.718622,0.049734,0.414861


In [28]:
BEST_T_FATAL = 0.6
BEST_T_SER   = 0.5

y_final = predict_with_thresholds(T_FATAL=BEST_T_FATAL, T_SERIOUS=BEST_T_SER)

print("Final model with tuned thresholds (T_FATAL=0.6, T_SERIOUS=0.5):")
print(classification_report(y_test, y_final))
print(confusion_matrix(y_test, y_final))


Final model with tuned thresholds (T_FATAL=0.6, T_SERIOUS=0.5):
              precision    recall  f1-score   support

           1       0.08      0.19      0.11       969
           2       0.18      0.18      0.18     11564
           3       0.89      0.88      0.88     87467

    accuracy                           0.79    100000
   macro avg       0.38      0.42      0.39    100000
weighted avg       0.80      0.79      0.79    100000

[[  183    52   734]
 [  605  2136  8823]
 [ 1503  9391 76573]]


In [ ]:
def predict_with_thresholds_on_X(model, X, T_FATAL=0.6, T_SERIOUS=0.5):

    proba = model.predict_proba(X)
    classes = list(model.named_steps["rf"].classes_)  # [1,2,3]

    idx_fatal   = classes.index(1)
    idx_serious = classes.index(2)
    idx_slight  = classes.index(3)

    fatal_p   = proba[:, idx_fatal]
    serious_p = proba[:, idx_serious]
    slight_p  = proba[:, idx_slight]

    y_custom = np.full(len(X), 3, dtype=int)  # default Slight

    for i in range(len(X)):
        if fatal_p[i] >= T_FATAL:
            y_custom[i] = 1
        elif serious_p[i] >= T_SERIOUS:
            y_custom[i] = 2
        else:
            y_custom[i] = 3

    return y_custom


In [30]:
BEST_T_FATAL = 0.6
BEST_T_SER   = 0.5

X_all = df[feature_cols]
y_all = df["casualty_severity"]

y_all_pred = predict_with_thresholds_on_X(
    model_rf3, X_all,
    T_FATAL=BEST_T_FATAL,
    T_SERIOUS=BEST_T_SER
)

from sklearn.metrics import classification_report, confusion_matrix

print("Full dataset (TRAIN + TEST together, not a clean eval):")
print(classification_report(y_all, y_all_pred))
print(confusion_matrix(y_all, y_all_pred))


Full dataset (TRAIN + TEST together, not a clean eval):
              precision    recall  f1-score   support

           1       0.09      0.20      0.12    109203
           2       0.19      0.19      0.19   1235185
           3       0.89      0.87      0.88   9348205

    accuracy                           0.79  10692593
   macro avg       0.39      0.42      0.40  10692593
weighted avg       0.80      0.79      0.79  10692593

[[  21482    5215   82506]
 [  60998  231912  942275]
 [ 164537 1011805 8171863]]
